# Climate Change Data Analysis - Machine Learning

This notebook performs comprehensive analysis of climate data including:
- Temperature anomaly trends
- CO₂ atmospheric concentrations
- Methane (CH₄) levels
- Global sea level rise

## Analysis includes:
1. Exploratory Data Analysis (EDA)
2. Trend Estimation
3. Time-Series Forecasting (ARIMA/Prophet)
4. Correlation Analysis
5. Predictive Modeling

## 1. Import Required Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Time series
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-darkgrid')

print("Libraries imported successfully!")

## 2. Load Climate Data

In [ ]:
# Load the climate dataset
df = pd.read_csv('../data/sample/climate_sample.csv')

# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nStatistical Summary:")
print(df.describe())

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

# Check data types
print("\nData Types:")
print(df.dtypes)

# Year range
print(f"\nYear Range: {df['year'].min()} to {df['year'].max()}")
print(f"Total Years: {df['year'].max() - df['year'].min() + 1}")

### 3.1 Temperature Anomaly Visualization

In [ ]:
# Create temperature anomaly visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['year'],
    y=df['global_temp_anomaly'],
    mode='lines+markers',
    name='Temperature Anomaly',
    line=dict(color='red', width=2),
    marker=dict(size=4)
))

fig.update_layout(
    title='Global Temperature Anomaly (1924-2023)',
    xaxis_title='Year',
    yaxis_title='Temperature Anomaly (°C)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

### 3.2 CO₂ Concentration Trend

In [ ]:
# CO2 concentration visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['year'],
    y=df['co2_ppm'],
    mode='lines',
    name='CO₂ Concentration',
    line=dict(color='green', width=2),
    fill='tozeroy',
    fillcolor='rgba(0,128,0,0.1)'
))

fig.update_layout(
    title='Atmospheric CO₂ Concentration (1924-2023)',
    xaxis_title='Year',
    yaxis_title='CO₂ (ppm)',
    hovermode='x unified',
    template='plotly_white',
    height=500
)

fig.show()

### 3.3 Combined Climate Indicators

In [ ]:
# Create subplot with all climate indicators
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Temperature Anomaly', 'CO₂ Concentration', 
                    'Methane Levels', 'Sea Level Rise')
)

# Temperature
fig.add_trace(
    go.Scatter(x=df['year'], y=df['global_temp_anomaly'], 
               name='Temp Anomaly', line=dict(color='red')),
    row=1, col=1
)

# CO2
fig.add_trace(
    go.Scatter(x=df['year'], y=df['co2_ppm'], 
               name='CO₂', line=dict(color='green')),
    row=1, col=2
)

# Methane
fig.add_trace(
    go.Scatter(x=df['year'], y=df['methane_ppb'], 
               name='Methane', line=dict(color='orange')),
    row=2, col=1
)

# Sea Level
fig.add_trace(
    go.Scatter(x=df['year'], y=df['sea_level_mm'], 
               name='Sea Level', line=dict(color='blue')),
    row=2, col=2
)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="ppm", row=1, col=2)
fig.update_yaxes(title_text="ppb", row=2, col=1)
fig.update_yaxes(title_text="mm", row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="Climate Indicators Dashboard")
fig.show()

## 4. Correlation Analysis

In [ ]:
# Calculate correlation matrix
features = ['global_temp_anomaly', 'co2_ppm', 'methane_ppb', 'sea_level_mm']
corr_matrix = df[features].corr()

# Create correlation heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=['Temp', 'CO₂', 'CH₄', 'Sea Level'],
    y=['Temp', 'CO₂', 'CH₄', 'Sea Level'],
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(3),
    texttemplate='%{text}',
    textfont={"size": 14},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Correlation Matrix of Climate Indicators',
    height=600,
    width=700
)

fig.show()

print("\nCorrelation Matrix:")
print(corr_matrix)

### 4.1 CO₂ vs Temperature Relationship

In [ ]:
# Scatter plot: CO2 vs Temperature
fig = px.scatter(df, x='co2_ppm', y='global_temp_anomaly', 
                 color='year', size='sea_level_mm',
                 title='CO₂ Concentration vs Temperature Anomaly',
                 labels={'co2_ppm': 'CO₂ (ppm)', 
                        'global_temp_anomaly': 'Temperature Anomaly (°C)'},
                 color_continuous_scale='Viridis',
                 height=600)

# Add trendline
from sklearn.linear_model import LinearRegression
X = df['co2_ppm'].values.reshape(-1, 1)
y = df['global_temp_anomaly'].values
model = LinearRegression().fit(X, y)
y_pred = model.predict(X)

fig.add_trace(go.Scatter(x=df['co2_ppm'], y=y_pred,
                         mode='lines',
                         name='Trend Line',
                         line=dict(color='red', width=3, dash='dash')))

fig.show()

print(f"CO₂-Temperature Correlation: {df['co2_ppm'].corr(df['global_temp_anomaly']):.4f}")
print(f"Linear Model R² Score: {r2_score(y, y_pred):.4f}")

## 5. Trend Analysis and Linear Regression

In [ ]:
# Temperature trend analysis
X = df['year'].values.reshape(-1, 1)
y = df['global_temp_anomaly'].values

# Fit linear regression model
temp_model = LinearRegression()
temp_model.fit(X, y)
y_pred = temp_model.predict(X)

# Calculate metrics
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))
slope = temp_model.coef_[0]
intercept = temp_model.intercept_

print("Temperature Trend Analysis")
print("=" * 50)
print(f"Slope (°C per year): {slope:.6f}")
print(f"Intercept: {intercept:.4f}")
print(f"R² Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"\nInterpretation: Temperature is {'increasing' if slope > 0 else 'decreasing'} at {abs(slope):.6f}°C per year")
print(f"Projected change over 100 years: {slope * 100:.3f}°C")

# Visualize
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['year'], y=y, mode='markers', name='Actual', marker=dict(color='blue')))
fig.add_trace(go.Scatter(x=df['year'], y=y_pred, mode='lines', name='Trend', line=dict(color='red', width=3)))
fig.update_layout(title='Temperature Anomaly: Linear Trend',
                  xaxis_title='Year', yaxis_title='Temperature Anomaly (°C)',
                  height=500)
fig.show()

### 5.1 CO₂ Growth Rate Analysis

In [ ]:
# CO2 trend analysis
X = df['year'].values.reshape(-1, 1)
y = df['co2_ppm'].values

co2_model = LinearRegression()
co2_model.fit(X, y)
y_pred_co2 = co2_model.predict(X)

slope_co2 = co2_model.coef_[0]
r2_co2 = r2_score(y, y_pred_co2)

print("CO₂ Concentration Trend Analysis")
print("=" * 50)
print(f"Growth rate: {slope_co2:.4f} ppm per year")
print(f"R² Score: {r2_co2:.4f}")
print(f"Projected increase over 100 years: {slope_co2 * 100:.1f} ppm")

# Calculate decade averages
df['decade'] = (df['year'] // 10) * 10
decade_avg = df.groupby('decade')['co2_ppm'].mean()

print("\nDecade Averages:")
print(decade_avg)

## 6. Time Series Forecasting

In [ ]:
# Simple linear extrapolation forecast
years_ahead = 30
last_year = df['year'].max()
future_years = np.arange(last_year + 1, last_year + years_ahead + 1).reshape(-1, 1)

# Temperature forecast
temp_forecast = temp_model.predict(future_years)

# CO2 forecast
co2_forecast = co2_model.predict(future_years)

# Create forecast dataframe
forecast_df = pd.DataFrame({
    'year': future_years.flatten(),
    'temp_forecast': temp_forecast,
    'co2_forecast': co2_forecast
})

print("Temperature Forecast (Next 30 years)")
print(forecast_df[['year', 'temp_forecast']].head(10))
print("\nCO₂ Forecast (Next 30 years)")
print(forecast_df[['year', 'co2_forecast']].head(10))

In [ ]:
# Visualize forecast
fig = make_subplots(rows=1, cols=2, subplot_titles=('Temperature Forecast', 'CO₂ Forecast'))

# Temperature
fig.add_trace(go.Scatter(x=df['year'], y=df['global_temp_anomaly'], 
                         mode='lines', name='Historical Temp', line=dict(color='blue')),
              row=1, col=1)
fig.add_trace(go.Scatter(x=forecast_df['year'], y=forecast_df['temp_forecast'],
                         mode='lines', name='Forecast Temp', line=dict(color='red', dash='dash')),
              row=1, col=1)

# CO2
fig.add_trace(go.Scatter(x=df['year'], y=df['co2_ppm'],
                         mode='lines', name='Historical CO₂', line=dict(color='green')),
              row=1, col=2)
fig.add_trace(go.Scatter(x=forecast_df['year'], y=forecast_df['co2_forecast'],
                         mode='lines', name='Forecast CO₂', line=dict(color='orange', dash='dash')),
              row=1, col=2)

fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="CO₂ (ppm)", row=1, col=2)
fig.update_layout(height=500, title_text="Climate Forecasts (30 Years)")
fig.show()

## 7. Multivariate Regression Model

In [ ]:
# Build multivariate model: Predict temperature from other factors
features = ['co2_ppm', 'methane_ppb', 'sea_level_mm']
target = 'global_temp_anomaly'

X = df[features].values
y = df[target].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
multi_model = LinearRegression()
multi_model.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = multi_model.predict(X_train_scaled)
y_test_pred = multi_model.predict(X_test_scaled)

# Evaluate
r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae = mean_absolute_error(y_test, y_test_pred)

print("Multivariate Regression Model Results")
print("=" * 50)
print(f"Features: {features}")
print(f"Target: {target}")
print(f"\nTraining R² Score: {r2_train:.4f}")
print(f"Testing R² Score: {r2_test:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

# Feature importance
print("\nFeature Coefficients (Standardized):")
for feature, coef in zip(features, multi_model.coef_):
    print(f"  {feature}: {coef:.4f}")

In [ ]:
# Visualize actual vs predicted
fig = go.Figure()

fig.add_trace(go.Scatter(x=y_test, y=y_test_pred, mode='markers',
                         name='Predictions', marker=dict(size=10, color='blue', opacity=0.6)))

# Add perfect prediction line
min_val = min(y_test.min(), y_test_pred.min())
max_val = max(y_test.max(), y_test_pred.max())
fig.add_trace(go.Scatter(x=[min_val, max_val], y=[min_val, max_val],
                         mode='lines', name='Perfect Prediction',
                         line=dict(color='red', dash='dash', width=2)))

fig.update_layout(
    title=f'Multivariate Model: Actual vs Predicted (R²={r2_test:.4f})',
    xaxis_title='Actual Temperature Anomaly (°C)',
    yaxis_title='Predicted Temperature Anomaly (°C)',
    height=600
)

fig.show()

## 8. Key Findings and Summary

In [ ]:
# Summary statistics
print("=" * 70)
print("CLIMATE CHANGE ANALYSIS SUMMARY (1924-2023)")
print("=" * 70)

print("\n1. TEMPERATURE TRENDS:")
print(f"   - Average anomaly: {df['global_temp_anomaly'].mean():.4f}°C")
print(f"   - Minimum anomaly: {df['global_temp_anomaly'].min():.4f}°C (Year: {df.loc[df['global_temp_anomaly'].idxmin(), 'year']:.0f})")
print(f"   - Maximum anomaly: {df['global_temp_anomaly'].max():.4f}°C (Year: {df.loc[df['global_temp_anomaly'].idxmax(), 'year']:.0f})")
print(f"   - Total change: {df['global_temp_anomaly'].max() - df['global_temp_anomaly'].min():.4f}°C")
print(f"   - Annual increase rate: {slope:.6f}°C/year")

print("\n2. CO₂ CONCENTRATIONS:")
print(f"   - Starting level (1924): {df.loc[df['year'] == 1924, 'co2_ppm'].values[0]:.1f} ppm")
print(f"   - Ending level (2023): {df.loc[df['year'] == 2023, 'co2_ppm'].values[0]:.1f} ppm")
print(f"   - Total increase: {df['co2_ppm'].max() - df['co2_ppm'].min():.1f} ppm")
print(f"   - Annual increase rate: {slope_co2:.4f} ppm/year")

print("\n3. METHANE LEVELS:")
print(f"   - Starting level: {df['methane_ppb'].min():.1f} ppb")
print(f"   - Ending level: {df['methane_ppb'].max():.1f} ppb")
print(f"   - Total increase: {df['methane_ppb'].max() - df['methane_ppb'].min():.1f} ppb")

print("\n4. SEA LEVEL RISE:")
print(f"   - Total rise: {df['sea_level_mm'].max() - df['sea_level_mm'].min():.1f} mm")
print(f"   - Current level: {df['sea_level_mm'].max():.1f} mm above baseline")

print("\n5. CORRELATIONS:")
print(f"   - CO₂ vs Temperature: {df['co2_ppm'].corr(df['global_temp_anomaly']):.4f}")
print(f"   - Methane vs Temperature: {df['methane_ppb'].corr(df['global_temp_anomaly']):.4f}")
print(f"   - Sea Level vs Temperature: {df['sea_level_mm'].corr(df['global_temp_anomaly']):.4f}")

print("\n6. FORECAST (30 YEARS):")
print(f"   - Projected temp in 2053: {forecast_df.loc[forecast_df['year'] == 2053, 'temp_forecast'].values[0]:.4f}°C")
print(f"   - Projected CO₂ in 2053: {forecast_df.loc[forecast_df['year'] == 2053, 'co2_forecast'].values[0]:.1f} ppm")

print("\n" + "=" * 70)